In [1]:
import numpy as np
import pandas as pd

In [2]:
orders = pd.read_csv("/content/drive/MyDrive/DA&AI/SILVER DATA/ORDERS.csv")
order_items = pd.read_csv("/content/drive/MyDrive/DA&AI/SILVER DATA/ORDER_ITEM.csv")
products = pd.read_csv("/content/drive/MyDrive/DA&AI/SILVER DATA/product.csv")

In [3]:
orders['order_date']=pd.to_datetime(orders['order_date'])
orders['yearmonth']=orders['order_date'].dt.to_period('M')

In [4]:
df_sales = pd.merge(order_items, orders, on='order_id', how='inner')
df_sales = pd.merge(df_sales, products, on='product_id', how='inner')

In [5]:
df_sales_2022 = df_sales[df_sales['order_date'].dt.year == 2022]
display(df_sales_2022.head())

,product_id,order_id,promo_id,quantity,unit_price,discount_amount,order_date,order_status,device_type,order_source,...,sales_employee_id,zip,yearmonth,product_name,category,segment,size,color,price,cogs
676211,449,787401,PROMO-0044,5,9981.10,9981.10,2022-01-01,delivered,mobile,referral,...,EMP0111,6413,2022-01,SaigonFlex UM-54,Streetwear,Balanced,M,yellow,12600.871631,11279.040197
676215,506,787404,PROMO-0044,4,12367.03,9893.62,2022-01-01,delivered,tablet,email_campaign,...,EMP0084,8344,2022-01,SaigonFlex UM-11,Streetwear,Balanced,L,green,15456.909753,14463.030456
676229,793,787423,PROMO-0044,3,804.05,482.43,2022-01-01,returned,mobile,referral,...,EMP0073,19342,2022-01,HanoiStreet RP-81,Outdoor,Activewear,M,blue,1012.756364,643.505394
676230,794,787423,PROMO-0044,6,768.99,922.79,2022-01-01,returned,mobile,referral,...,EMP0073,19342,2022-01,HanoiStreet RP-82,Outdoor,Activewear,L,white,1012.756364,962.118546
676235,494,787429,PROMO-0044,4,15032.88,12026.30,2022-01-01,delivered,mobile,organic_search,...,EMP0126,24521,2022-01,SaigonFlex UM-99,Streetwear,Balanced,L,white,19351.386393,13062.185816


In [6]:
df_sales[['quantity', 'unit_price', 'discount_amount']] = df_sales[['quantity', 'unit_price', 'discount_amount']].fillna(0)

Problem statement 1: Tính tổng doanh thu thực tế (sau chiết khấu)

In [7]:
df_sales['total_sales'] = df_sales['quantity'] * df_sales['unit_price'] * (1 - df_sales['discount_amount'])

In [8]:
total_actual_revenue = df_sales['total_sales'].sum()

print(f"Tổng doanh thu thực tế (Total Actual Revenue): {total_actual_revenue:,.2f}")

Tổng doanh thu thực tế (Total Actual Revenue): -29,155,930,281,893.00


Problem Statement 2: So sánh tỷ trọng đóng góp doanh thu của các danh mục sản phẩm theo tháng

In [13]:
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
monthly_category_rev = df_sales.groupby(['yearmonth', 'category'])['total_sales'].sum().reset_index()

monthly_total_rev = monthly_category_rev.groupby('yearmonth')['total_sales'].sum().reset_index()
monthly_total_rev.rename(columns={'total_sales': 'Total_Monthly_Revenue'}, inplace=True)

contribution_df = pd.merge(monthly_category_rev, monthly_total_rev, on='yearmonth')
contribution_df['Contribution_Pct (%)'] = (contribution_df['total_sales'] / contribution_df['Total_Monthly_Revenue']) * 100


In [18]:
pivot_contribution = contribution_df.pivot_table(
    index='category',
    columns='yearmonth',
    values='Contribution_Pct (%)',
    fill_value=0
)

print(pivot_contribution.round(2))

yearmonth   2012-07  2012-08  2012-09  2012-10  2012-11  2012-12  2013-01  \
category                                                                    
Casual         1.83     1.64     1.42     1.65     1.52     1.39    -0.74   
GenZ           1.73     1.59     1.29     1.09     0.72     1.00    -0.29   
Outdoor       13.59    16.47    20.23    20.16    25.78    38.98   128.74   
Streetwear    82.85    80.30    77.06    77.10    71.98    58.62   -27.71   

yearmonth   2013-02  2013-03  2013-04  ...  2022-03  2022-04  2022-05  \
category                               ...                              
Casual        -0.00     1.08     1.40  ...     3.05     3.09     4.27   
GenZ          -0.00     0.46     0.50  ...     0.66     0.84     2.82   
Outdoor      100.17     7.36     5.70  ...     3.53     3.03     7.12   
Streetwear    -0.17    91.11    92.41  ...    92.76    93.04    85.79   

yearmonth   2022-06  2022-07  2022-08  2022-09  2022-10  2022-11  2022-12  
category              